# RQ4 — ViGoEmotions noise robustness

Use the frozen 28-label PhoBERT checkpoint and recorded RQ1 thresholds. Attach the RQ4 code ZIP, the original private 28-label model archive, and authorized CSV inputs. Select a GPU accelerator. Run validation first. The final cell runs held-out evaluation only when enabled.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile

INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
ROOT = Path('/tmp/vigoemotions_rq4').resolve()
CONFIG = ROOT / 'configs/rq4_noise_robustness.json'
if not ROOT.exists():
    archives = list(INPUT.rglob('vigoemotions_rq4_code.zip'))
    expanded = [p.parent.parent for p in INPUT.rglob('rq4_robustness.py') if p.parent.name == 'scripts']
    if len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as archive:
            if any(not (ROOT / path).resolve().is_relative_to(ROOT) for path in archive.namelist()):
                raise ValueError('Code ZIP contains an invalid path')
            archive.extractall(ROOT)
    elif not archives and len(expanded) == 1:
        shutil.copytree(expanded[0], ROOT)
    else:
        raise RuntimeError('Attach exactly one RQ4 code bundle')
if not CONFIG.is_file():
    raise RuntimeError('RQ4 code workspace is incomplete')
print(ROOT)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(ROOT / 'requirements-colab.txt')], check=True)
if shutil.which('java') is None:
    raise RuntimeError('Enable Java for the original VnCoreNLP word segmenter')

In [ ]:
rq1_config = json.loads((ROOT / 'configs/final_thresholds.json').read_text())
rq1_lock = json.loads((ROOT / 'reports/final28/pre_test_lock.json').read_text())
source_manifest = json.loads((ROOT / rq1_config['model']['source_manifest']).read_text())
DATA = ROOT / 'data'
DATA.mkdir(exist_ok=True)

def checksums(raw):
    lf = raw.replace(b'\r\n', b'\n')
    return {hashlib.sha256(data).hexdigest() for data in (raw, lf, lf.replace(b'\n', b'\r\n'))}

def stage_csv(split):
    target = DATA / f'{split}.csv'
    expected = rq1_lock['sources_sha256'].get(f'private_source/{split}.csv',
                                            source_manifest['environment']['data_sha256'][split])

    def matches(path):
        raw = path.read_bytes()
        if split == 'test':
            return expected in checksums(raw)
        return hashlib.sha256(raw).hexdigest() == expected

    if target.exists() and matches(target):
        return
    candidates = [p for p in sorted(INPUT.rglob(f'{split}.csv')) if matches(p)]
    if not candidates:
        raise FileNotFoundError(f'Attach the original authorized {split}.csv matching the RQ1 checksum')
    shutil.copyfile(candidates[0], target)

for split in ('train', 'val'):
    stage_csv(split)

def run(script, *args):
    subprocess.run([sys.executable, str(ROOT / 'scripts' / script), *args], cwd=ROOT, check=True)

name = Path(rq1_config['model']['archive_path'])
if not (DATA / 'final28/assets.json').is_file():
    archives = sorted(INPUT.rglob(name.name))
    if not archives:
        archives = sorted(INPUT.rglob(name.with_suffix('.bin').name))
    if len(archives) != 1:
        raise RuntimeError('Attach exactly one intact private 28-label checkpoint archive')
    run('final_thresholds.py', 'restore', '--archive', str(archives[0]))
run('rq4_robustness.py', 'verify')

In [ ]:
OUTPUT = ROOT / 'reports/rq4_noise'
if not (OUTPUT / 'val/manifest.json').is_file():
    run('rq4_robustness.py', 'validate', '--device', 'cuda')
archive = shutil.make_archive(str(WORKING / 'vigoemotions_rq4_val'), 'zip', OUTPUT, 'val')
print(archive)

Review the validation `tables/noise_coverage.csv`, `tables/comparison.csv`, and `manifest.json`. Then enable the cell below to stage the authorized test CSV and evaluate the paired test conditions. Use the same environment as validation.

In [ ]:
RUN_HELD_OUT = False
if RUN_HELD_OUT:
    stage_csv('test')
    if not (OUTPUT / 'test/manifest.json').is_file():
        run('rq4_robustness.py', 'test', '--device', 'cuda')
    archive = shutil.make_archive(str(WORKING / 'vigoemotions_rq4_test'), 'zip', OUTPUT, 'test')
    print(archive)
else:
    print('Held-out evaluation is disabled in this cell.')